In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

c:\Users\disha\OneDrive\Desktop\Langchain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\disha\AppData\Local\Temp\ipykernel_20940\3754471924.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [5]:
# 1. Load documents
loader = TextLoader("game.txt")
documents = loader.load()

In [6]:
# 2. Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(documents)

In [7]:
# 3. Create Hugging Face embeddings (runs locally via sentence-transformers)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},   # use "cuda" if you have a GPU
    encode_kwargs={"normalize_embeddings": True}
    
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1478.07it/s]


In [8]:
# 4. Create vector store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

In [9]:
# 5. Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

In [10]:
# 6. Query
query = "What is the main topic of this document?"
results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(doc.page_content[:300])
    print(doc.metadata)

--- Result 1 ---
Oh boy, video games are so fun. 
So fun, I think Iâ€™ll play another one.

To some people, video games may be boring. 
But to me, theyâ€™re a world of exploring. 

Sometimes, video games make you rage. 
Thatâ€™s when we must learn to stop and flip the page. 

But we donâ€™t stop playing even when we
{'source': 'game.txt'}
